In [30]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch
import pprint
import rerun as rr
from types import SimpleNamespace

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools, qwen_utils_3d
from autoencoder.model_qwen import QwenAutoencoder
from benchmark.spatial_3d import (
    generate_with_vision_features_3d,
    compute_gaussian_grid_thw,
    create_gaussian_to_grid_mapping
)
from extract_graphs import decode_qwen

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
graph_dir = Path("output/aenoisy_states15_iter10k/video12_19500/graph")
preprocessed_dir = Path("data/preprocessed/qwen3_da3_subsampled/video12_19500")
ae_path = Path("data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_noise/best_ckpt.pth")

print(f"Graph dir: {graph_dir}")
print(f"Autoencoder path: {ae_path}")

Graph dir: output/aenoisy_states15_iter10k/video12_19500/graph
Autoencoder path: data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_noise/best_ckpt.pth


In [7]:
# Load autoencoder
autoencoder = QwenAutoencoder(input_dim=20480, latent_dim=16).to("cuda")
autoencoder.load_state_dict(torch.load(ae_path, map_location="cuda"))
autoencoder.eval()
print(f"Autoencoder loaded from {ae_path}")

# Load graph data and create toolkit (accepts npz directly)
toolkit = tools.GraphTools(
    positions=np.load(graph_dir / "positions.npy"),
    clusters=np.load(graph_dir / "clusters.npy"),
    centroids=np.load(graph_dir / "c_centroids.npy"),
    centers=np.load(graph_dir / "c_centers.npy"),
    extents=np.load(graph_dir / "c_extents.npy"),
    adjacency=np.load(graph_dir / "graph.npy"),
    bhattacharyya_coeffs=np.load(graph_dir / "bhattacharyya_coeffs.npy"),
    qwen_feats=np.load(graph_dir / "c_qwen_feats.npz"),
    patch_latents_through_time=np.load(graph_dir / "patch_latents_through_time.npy"),
    autoencoder=autoencoder,
)
graph_tools = toolkit.get_all_tools()
print(f"Loaded {len(graph_tools)} tools: {list(graph_tools.keys())}")
print(f"Graph has {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

Autoencoder loaded from data/preprocessed/qwen3_da3_subsampled/video12_19500/autoencoder_noise/best_ckpt.pth
Loaded 9 tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'voxelize_scene', 'node_movement', 'relative_movement']
Graph has 10 nodes, 20 timesteps


In [5]:
model, processor = qwen_utils_3d.get_custom_qwen3_3d(size="32B", use_fp8=False)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model-00003-of-00014.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00001-of-00014.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [61]:
rr.init("grid tmp")
rr.save("gaussian_grid_tmp.rrd")

def log_grid_and_gaussians(prefix, pos, patch_size, min_x, min_y, image_grid_thw):

    gaussian_means_sampled_projected_sorted = pos[:, :2]

    sampled_means_xy = np.column_stack(
        (gaussian_means_sampled_projected_sorted[:, 0], gaussian_means_sampled_projected_sorted[:, 1], np.zeros(len(gaussian_means_sampled_projected_sorted)))
    )
    rr.set_time_sequence("frame", 0)
    rr.log(
        f"world/{prefix}_gaussians",
        rr.Points3D(
            positions=sampled_means_xy,
            colors=np.array([0, 0, 255]),
        ),
    )
    # For all points on the grid, put a red point in rerun
    grid_positions = torch.tensor([[i * patch_size + min_x, j * patch_size + min_y, 0] for i in range(image_grid_thw[0, 2]) for j in range(image_grid_thw[0, 1])])
    rr.log(
        f"world/{prefix}_grid",
        rr.Points3D(
            positions=grid_positions,
            colors=torch.tensor([255, 0, 0]),
        ),
    )

In [62]:
hook = 0
gallbladder = 2
liver = 9

hook_mask = toolkit.clusters == hook
gallbladder_mask = toolkit.clusters == gallbladder
liver_mask = toolkit.clusters == liver

hook_feats = toolkit.patch_latents_through_time[0][hook_mask]
gallbladder_feats = toolkit.patch_latents_through_time[0][gallbladder_mask]
liver_feats = toolkit.patch_latents_through_time[0][liver_mask]

hook_positions = toolkit.positions[0][hook_mask]
gallbladder_positions = toolkit.positions[0][gallbladder_mask]
liver_positions = toolkit.positions[0][liver_mask]

In [64]:
feat = hook_feats
pos = hook_positions

mock_cfg = SimpleNamespace(
    graph_extraction=SimpleNamespace(
        decode_batch_size=1024
    )
)
feat = decode_qwen(autoencoder, torch.tensor(feat, device=model.device).float(), mock_cfg)
feat = torch.tensor(feat, device=model.device).float()

PATCH_SIZE = 0.01

image_grid_thw, min_x, min_y = compute_gaussian_grid_thw(gaussian_means=pos, patch_size=PATCH_SIZE)
gaussian_to_grid_mapping, sorted_indices = create_gaussian_to_grid_mapping(gaussian_means=pos, image_grid_thw=image_grid_thw, patch_size=PATCH_SIZE)

feat = feat[sorted_indices]
pos = pos[sorted_indices]
gaussian_to_grid_mapping = gaussian_to_grid_mapping[sorted_indices]

# print("image_grid_thw, min_x, min_y")
# print(image_grid_thw, min_x, min_y)
# print("pos.min(axis=0), pos.max(axis=0)")
# print(pos.min(axis=0), pos.max(axis=0))
# print("gaussian_to_grid_mapping")
# print(gaussian_to_grid_mapping)

log_grid_and_gaussians("unfiltered", pos, PATCH_SIZE, min_x, min_y, image_grid_thw)
print(len(gaussian_to_grid_mapping), len(np.unique(gaussian_to_grid_mapping, axis=0)))


2026-02-17 14:14:45.511 | INFO     | benchmark.spatial_3d:compute_gaussian_grid_thw:58 - Scene min/max: min_x=-0.06179437413811684, min_y=-0.1811622977256775, max_x=0.14135083556175232, max_y=-0.04818115755915642
2026-02-17 14:14:45.511 | INFO     | benchmark.spatial_3d:compute_gaussian_grid_thw:64 - Scene extent: extent_x=0.20314520597457886, extent_y=0.13298113644123077
2026-02-17 14:14:45.512 | INFO     | benchmark.spatial_3d:compute_gaussian_grid_thw:74 - Image grid thw: tensor([[ 1, 15, 22]])


1801 111


In [59]:
messages = [
    {"role": "system", "content": [{"type": "text", "text": "Your are a helpful assistant."}]},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Image:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "\n\nWhat do you see in this image?"},
        ],
    },
]

generate_with_vision_features_3d(
    messages=messages,
    vision_features=[feat],
    model=model,
    processor=processor,
    image_grid_thw=image_grid_thw,
    gaussian_to_grid_mapping=gaussian_to_grid_mapping,
    gaussians_per_image=[len(feat)],
    zero_positional_encodings=False,
)

2026-02-17 14:12:11.875 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:207 - ================================================================================
2026-02-17 14:12:11.876 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:208 - PREPROCESSING INPUTS
2026-02-17 14:12:11.877 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:209 - ================================================================================
2026-02-17 14:12:11.878 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:213 - image_token_id: 151655
2026-02-17 14:12:11.879 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:216 - image_pad_token_id: 151655
2026-02-17 14:12:11.879 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:219 - vision_start_token_id: 151652
2026-02-17 14:12:11.880 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:222 - vision_end_token_id: 151653
2026-02-17 14:12:11.880 | INFO

Decoded input: <|im_start|>system
Your are a helpful assistant.<|im_end|>
<|im_start|>user
Image:<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|i

2026-02-17 14:12:30.288 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:298 - ================================================================================
2026-02-17 14:12:30.288 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:299 - GENERATION COMPLETE
2026-02-17 14:12:30.289 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:300 - ================================================================================
2026-02-17 14:12:30.290 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:301 - Input length: 1835 tokens
2026-02-17 14:12:30.291 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:302 - Generated length: 2272 tokens
2026-02-17 14:12:30.292 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:303 - New tokens generated: 437
2026-02-17 14:12:30.292 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:304 - max_new_tokens limit: 1000
2026-02-17 14:12:30.294 | INFO

"So, let's look at the image. It's a close-up of what seems to be a surgical procedure, probably laparoscopic. There are multiple instruments—those cylindrical, gray tubes with white tips. They're inserted into the body, so that's typical for laparoscopy where they use trocars. The background shows internal tissues: there's a yellowish fatty tissue, some red and purple areas which are probably organs or blood vessels. The tissue has some white, maybe scar tissue or the area being worked on. The instruments are aligned in a way that suggests they're manipulating or cutting tissue. The image is from inside the body, so it's an endoscopic or laparoscopic view. I need to describe what's visible: the surgical instruments, the internal anatomy with different colored tissues (fatty, vascular, maybe peritoneum), and the context of a minimally invasive surgery.\n</think>\n\nThis image depicts a **laparoscopic surgical procedure** (a minimally invasive surgery performed using small incisions and

In [65]:
def top_opacity_per_cell(opacities, gaussian_to_grid_mapping):
    mask = np.zeros(len(gaussian_to_grid_mapping), dtype=bool)
    unique_hw = np.unique(gaussian_to_grid_mapping, axis=0)
    for hw in unique_hw:
        hw_indices = np.where((gaussian_to_grid_mapping == hw).all(axis=1))[0]
        cell_opacities = opacities[hw_indices]
        hw_highest_opacity_idx = hw_indices[np.argmax(cell_opacities)]
        mask[hw_highest_opacity_idx] = True
    return mask
    
opacities = np.load(graph_dir / "opacities_through_time.npy")[0]
top_per_cell_mask = top_opacity_per_cell(opacities, gaussian_to_grid_mapping)
feat_topo = feat[top_per_cell_mask]
pos_topo = pos[top_per_cell_mask]
gaussian_to_grid_mapping_topo = gaussian_to_grid_mapping[top_per_cell_mask]

log_grid_and_gaussians("top_opacity", pos_topo, PATCH_SIZE, min_x, min_y, image_grid_thw)


In [66]:
messages = [
    {"role": "system", "content": [{"type": "text", "text": "Your are a helpful assistant."}]},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Image:"},
            {"type": "image", "image": None},
            {"type": "text", "text": "\n\nWhat do you see in this image?"},
        ],
    },
]

generate_with_vision_features_3d(
    messages=messages,
    vision_features=[feat_topo],
    model=model,
    processor=processor,
    image_grid_thw=image_grid_thw,
    gaussian_to_grid_mapping=gaussian_to_grid_mapping_topo,
    gaussians_per_image=[len(feat_topo)],
    zero_positional_encodings=False,
)

2026-02-17 14:16:15.236 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:207 - ================================================================================
2026-02-17 14:16:15.236 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:208 - PREPROCESSING INPUTS
2026-02-17 14:16:15.237 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:209 - ================================================================================
2026-02-17 14:16:15.237 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:213 - image_token_id: 151655
2026-02-17 14:16:15.238 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:216 - image_pad_token_id: 151655
2026-02-17 14:16:15.238 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:219 - vision_start_token_id: 151652
2026-02-17 14:16:15.238 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:222 - vision_end_token_id: 151653
2026-02-17 14:16:15.239 | INFO

Decoded input: <|im_start|>system
Your are a helpful assistant.<|im_end|>
<|im_start|>user
Image:<|vision_start|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|image_pad|><|i

2026-02-17 14:16:35.049 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:298 - ================================================================================
2026-02-17 14:16:35.050 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:299 - GENERATION COMPLETE
2026-02-17 14:16:35.051 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:300 - ================================================================================
2026-02-17 14:16:35.051 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:301 - Input length: 145 tokens
2026-02-17 14:16:35.052 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:302 - Generated length: 622 tokens
2026-02-17 14:16:35.053 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:303 - New tokens generated: 477
2026-02-17 14:16:35.054 | INFO     | benchmark.spatial_3d:generate_with_vision_features_3d:304 - max_new_tokens limit: 1000
2026-02-17 14:16:35.055 | INFO  

"So, let's look at the image. It's a close-up of what seems to be a surgical procedure, probably laparoscopic. There are two instruments, maybe trocars or endoscopic tools, inserting into tissue. The tissue has different colors: yellowish (maybe fat), reddish (muscle or blood vessels), and some white areas. There's a lot of detail in the tissue, like blood vessels and the texture of the organ. The instruments are metallic, cylindrical, with one having a white tip. The background is the internal body, so this is likely from an abdominal surgery, maybe a procedure like a cholecystectomy (gallbladder removal) or something similar. Need to describe what's visible: surgical instruments, internal tissues, colors, and the context of a minimally invasive surgery.\n</think>\n\nThis image appears to be a close - up view from a **minimally invasive surgical procedure**, likely laparoscopic surgery. Here's what can be observed:  \n\n- **Surgical instruments**: Two cylindrical, metallic - looking t